In [1]:
!rm ita-eng.zip &> /dev/null
!rm rm -rf dataset &> /dev/null
!wget "https://www.manythings.org/anki/ita-eng.zip" &> /dev/null
!unzip "ita-eng.zip" -d "dataset" &> /dev/null

In [ ]:
! pip install transformers datasets evaluate accelerate sacrebleu
! pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [17]:
# prompt: Make a function that reads the file "text_file" and convert it into a dataset, with training, val and test sets, consider the the structure of the file is X\tY\tZ, X is the input and Y is the expected output
text_file = "./dataset/ita.txt"

import io
import random
from sklearn.model_selection import train_test_split
from datasets import Dataset

SAMPLES = 166_667
def create_dataset(file_path, test_size=0.2, val_size=0.25):
    """
    Reads a file with X\tY\tZ structure, where X is the input and Y is the expected output,
    and converts it into a dataset with training, validation, and test sets.

    Args:
        file_path (str): The path to the text file.
        test_size (float): The proportion of the dataset to include in the test split.
        val_size (float): The proportion of the remaining data to include in the validation split.

    Returns:
        tuple: A tuple containing the train_dataset, val_dataset, and test_dataset.
    """
    inputs = []
    outputs = []

    with io.open(file_path, encoding='UTF-8') as f:
        lines = f.read().strip().split('\n')

    for line in lines[:SAMPLES]:
        parts = line.split('\t')
        if len(parts) >= 2:
            inputs.append(parts[0])
            outputs.append(parts[1])

    # Split into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(inputs, outputs, test_size=test_size, random_state=42)

    # Split the training set into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=val_size, random_state=42)

    # Create Hugging Face Dataset objects
    train_dataset = Dataset.from_dict({"input": X_train, "output": y_train})
    val_dataset = Dataset.from_dict({"input": X_val, "output": y_val})
    test_dataset = Dataset.from_dict({"input": X_test, "output": y_test})

    return train_dataset, val_dataset, test_dataset

train_dataset, val_dataset, test_dataset = create_dataset(text_file)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

Train dataset size: 99999
Validation dataset size: 33334
Test dataset size: 33334


In [18]:
# prompt: make the tokenizer for the model which will be t5-small and model definition

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_checkpoint = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

In [19]:
# prompt: DO the data processing part, so convert the data effectevily with the tokenizer

max_input_length = 128
max_output_length = 128

input_prefix = "translate english to italian:"

def preprocess_function(examples):
    model_inputs = tokenizer([input_prefix + ex for ex in examples["input"]], max_length=max_input_length, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["output"], max_length=max_output_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
tokenized_val_dataset = val_dataset.map(preprocess_function, batched=True)
tokenized_test_dataset = test_dataset.map(preprocess_function, batched=True)

print(tokenized_train_dataset)
print(tokenized_val_dataset)
tokenized_test_dataset

Map:   0%|          | 0/99999 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3959: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/33334 [00:00<?, ? examples/s]

Map:   0%|          | 0/33334 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'output', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 99999
})
Dataset({
    features: ['input', 'output', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 33334
})


Dataset({
    features: ['input', 'output', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 33334
})

In [20]:
# prompt: Define the arguments for the training of the model

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=True,
)

In [21]:
# prompt: Define Metrics for Evaluation

import numpy as np
import evaluate

metric = evaluate.load("bleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    vocab_size = tokenizer.vocab_size
    # Replace predicted token IDs that are out of the vocabulary range with a valid token (e.g., pad token ID)
    # This prevents the OverflowError during decoding
    preds[preds >= vocab_size] = tokenizer.pad_token_id
    preds[preds < 0] = tokenizer.pad_token_id


    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"score": result["bleu"]}
    return result


In [22]:
# prompt: Initialize and Run the Trainer:
# Seq2SeqTrainer: Create an instance of Seq2SeqTrainer, passing your loaded model, training arguments, tokenized datasets (train and validation), data collator, and compute_metrics function.
# Start Training: Call trainer.train(). The Trainer will then manage the entire fine-tuning loop, including iterating through epochs, calculating loss, performing backpropagation, updating weights, and evaluating on the validation set at specified intervals.

from transformers import DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainer
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Score
1,2.139500,1.706661,0.103663
2,1.749500,1.408296,0.167142
3,1.634000,1.288494,0.195574
4,1.572600,1.256011,0.203905


TrainOutput(global_step=25000, training_loss=1.9250100732421875, metrics={'train_runtime': 5468.6799, 'train_samples_per_second': 73.143, 'train_steps_per_second': 4.571, 'total_flos': 1597363417350144.0, 'train_loss': 1.9250100732421875, 'epoch': 4.0})

In [23]:
# prompt: Evaluation and Saving:
# Test Set Evaluation: After training, evaluate your fine-tuned model on the separate test set using trainer.evaluate() to get an unbiased performance score (e.g., final BLEU score).
# Save Model: Save the fine-tuned model and tokenizer using trainer.save_model() and tokenizer.save_pretrained(). This allows you to load and use your custom model later for inference without retraining.

# Evaluate on the test set
test_results = trainer.evaluate(tokenized_test_dataset)
print("Test set evaluation results:", test_results)

# Save the fine-tuned model and tokenizer
model_save_path = "./fine_tuned_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"Model and tokenizer saved to {model_save_path}")

Test set evaluation results: {'eval_loss': 1.2620513439178467, 'eval_score': 0.20326758801252293, 'eval_runtime': 602.4055, 'eval_samples_per_second': 55.335, 'eval_steps_per_second': 3.459, 'epoch': 4.0}
Model and tokenizer saved to ./fine_tuned_model


In [29]:
# prompt: Make a think that is similar to def translate(sentence):
#             inputs = tokenizer(prefix + sentence, return_tensors="tf").input_ids
#             outputs = model.generate(inputs)
#             result = tokenizer.decode(outputs[0], skip_special_tokens=True)
#             return result
# and is used for different once and use it to translate and calculate the BLEU of the test set, also add some strings/things translates

def translate_and_evaluate(trainer, test_dataset, tokenizer, input_prefix="translate english to italian:"):
    """
    Translates sentences using the trained model and calculates the BLEU score on the test set.

    Args:
        trainer (Seq2SeqTrainer): The trained Seq2SeqTrainer object.
        test_dataset (Dataset): The test dataset.
        tokenizer (AutoTokenizer): The tokenizer used for the model.
        input_prefix (str): The prefix to add to input sentences for translation.

    Returns:
        tuple: A tuple containing the BLEU score and a list of dictionaries with original, translated, and reference sentences.
    """
    predictions = trainer.predict(test_dataset)

    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions.predictions, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(predictions.label_ids != -100, predictions.label_ids, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Post-process text
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    # Calculate BLEU score
    bleu_result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    print(bleu_result)
    bleu_score = bleu_result["bleu"]

    # Prepare examples for display
    translation_examples = []
    for original, translated, reference in zip(test_dataset["input"], decoded_preds, decoded_labels):
        translation_examples.append({
            "original": original,
            "translated": translated,
            "reference": reference
        })

    return bleu_score, translation_examples

bleu_score, translation_examples = translate_and_evaluate(trainer, tokenized_test_dataset, tokenizer, input_prefix=input_prefix)

print(f"BLEU score on the test set: {bleu_score}")

print("\nTranslation Examples:")
for example in translation_examples[:10]: # Display the first 10 examples
    print(f"Original: {example['original']}")
    print(f"Translated: {example['translated']}")
    print(f"Reference: {example['reference']}")
    print("-" * 20)


{'bleu': 0.20326758801252293, 'precisions': [0.5536172899287146, 0.26126042846248065, 0.14692164179104478, 0.08457101544941613], 'brevity_penalty': 0.9872348649157557, 'length_ratio': 0.9873156502847322, 'translation_length': 163147, 'reference_length': 165243}
BLEU score on the test set: 0.20326758801252293

Translation Examples:
Original: Call them this evening.
Translated: Chiave loro questa stanza.
Reference: Le chiami questa sera.
--------------------
Original: She has gone shopping.
Translated: Lei è andato a shopping.
Reference: Lei è andata a fare acquisti.
--------------------
Original: Maybe Tom left early.
Translated: Peutesse Tom è lasciato a premato.
Reference: Forse Tom se n'è andato presto.
--------------------
Original: Antibiotics are overused.
Translated: Les antibiotiques è suprautilizati.
Reference: Gli antibiotici vengono usati eccessivamente.
--------------------
Original: Why didn't you use it?
Translated: Perché non l'ha utto?
Reference: Perché non l'hai usata?


In [28]:
# You can also translate some custom strings here
custom_strings = [
    "Hello, how are you?",
    "This is a test sentence.",
    "I love learning new languages."
]

print("\nCustom String Translations:")
for text in custom_strings:
    # Tokenize the input string and get input IDs as a PyTorch tensor
    inputs = tokenizer(input_prefix + text, return_tensors="pt").input_ids
    # Move the input tensor to the same device as the model
    inputs = inputs.to(trainer.model.device)
    # Generate the translation
    outputs = trainer.model.generate(inputs)
    # Decode the generated token IDs back to a string
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Original: {text}")
    print(f"Translated: {result}")
    print("-" * 20)


Custom String Translations:
Original: Hello, how are you?
Translated: Ho, come siete?
--------------------
Original: This is a test sentence.
Translated: Questa è una stanza a testo.
--------------------
Original: I love learning new languages.
Translated: Amo appointare della nuova lingua.
--------------------

Custom String Translations:
Original: Hello, how are you?
Translated: Ho, come siete?
--------------------
Original: This is a test sentence.
Translated: Questa è una stanza a testo.
--------------------
Original: I love learning new languages.
Translated: Amo appointare della nuova lingua.
--------------------
